# 04 · Gold: star schema + county-day fact table
* `gold_dim_county`, `gold_dim_date` – dimensions
* `gold_fact_county_day` – one row per California county per local date, with fire activity, PM2.5, weather
  and data-quality flags. Every county gets a row every day, so "no fires" is an explicit 0 and missing PM2.5 is NULL.
* `gold_kpi_state_day` – statewide daily KPIs for the dashboard.

Gold does not limit processing to the original 2020 study window. Normal
incremental runs process every date queued by Silver, and `rebuild_all=true`
derives its full date range from the data currently present in the lakehouse.

The smoke indicator is descriptive, **not** a causal or regulatory estimate of wildfire-attributed PM2.5.

In [0]:
%run ./00_config

# 00 · Config
Shared settings for every notebook. Every other notebook loads this one with `%run ./00_config`.

**Before the first run:** change `CATALOG` / `SCHEMA` if needed, then run `01_setup` once to create the schema and volume.

Config loaded: workspace.wildfire, raw files in /Volumes/workspace/wildfire/raw


In [0]:
dbutils.widgets.dropdown(
    "rebuild_all", "false", ["false", "true"],
    "Rebuild all available dates"
)
REBUILD_ALL = dbutils.widgets.get("rebuild_all") == "true"
NEAR_KM = 100.0     # fire detections within this distance of a county's centre count as "nearby"
PM25_DAILY_STANDARD = 35.0   # EPA 24-hour PM2.5 standard, ug/m3

started = time.time()

# Silver queues every affected date. Do not constrain this list by START_DATE or
# END_DATE: those values describe the original replay configuration, not a Gold
# retention boundary.
pending_days = []
if spark.catalog.tableExists(t("ops_gold_pending_dates")):
    pending_days = [
        r.date_local
        for r in spark.table(t("ops_gold_pending_dates"))
        .select("date_local")
        .distinct()
        .collect()
    ]

# Get the available date bounds from the actual data. START_DATE and END_DATE
# are only fallbacks for an empty installation; they are not applied as filters.
def date_bounds_in_table(table_name):
    if not spark.catalog.tableExists(t(table_name)):
        return None, None
    if "date_local" not in spark.table(t(table_name)).columns:
        return None, None
    row = spark.table(t(table_name)).agg(
        F.min("date_local").alias("min_date"),
        F.max("date_local").alias("max_date"),
    ).first()
    return row["min_date"], row["max_date"]

source_tables = [
    "gold_fact_county_day",
    "silver_fire_detections",
    "silver_pm25_daily",
    "silver_weather_obs",
]
available_min = None
available_max = None
for table_name in source_tables:
    table_min, table_max = date_bounds_in_table(table_name)
    if table_min is not None:
        available_min = table_min if available_min is None else min(available_min, table_min)
    if table_max is not None:
        available_max = table_max if available_max is None else max(available_max, table_max)

if pending_days:
    pending_min = min(pending_days)
    pending_max = max(pending_days)
    available_min = pending_min if available_min is None else min(available_min, pending_min)
    available_max = pending_max if available_max is None else max(available_max, pending_max)

if available_min is None or available_max is None:
    available_min, available_max = START_DATE, END_DATE

if REBUILD_ALL:
    days = [
        available_min + dt.timedelta(days=i)
        for i in range((available_max - available_min).days + 1)
    ]
else:
    days = sorted(set(pending_days))

print(f"Rebuilding {len(days)} date(s)")
if days:
    print(f"Fact dates: {min(days)} through {max(days)}")

dimension_start = available_min
dimension_end = available_max
print(f"Date dimension: {dimension_start} through {dimension_end}")

Rebuilding 57 date(s)
Fact dates: 2020-08-15 through 2020-10-10
Date dimension: 2020-08-15 through 2020-10-10


## Dimensions

In [0]:
spark.table(t("silver_dim_county")).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(t("gold_dim_county"))

(
    spark.sql(
        f"SELECT explode(sequence(DATE'{dimension_start}', DATE'{dimension_end}', INTERVAL 1 DAY)) AS date_local"
    )
    .withColumn("date_key", F.date_format("date_local", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date_local"))
    .withColumn("month", F.month("date_local"))
    .withColumn("day_of_week", F.date_format("date_local", "E"))
    .withColumn("is_weekend", F.dayofweek("date_local").isin(1, 7))
    .withColumn("study_week", (F.datediff("date_local", F.lit(dimension_start)) / 7).cast("int") + 1)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(t("gold_dim_date"))
)

## Fact table for pending and requested dates

In [0]:
if days:
    county = spark.table(t("gold_dim_county"))
    grid = county.select("county_fips", "c_lat", "c_lon").crossJoin(
        spark.createDataFrame([(d,) for d in days], "date_local date")
    )

    fires = spark.table(t("silver_fire_detections")).where(F.col("date_local").isin(days))

    # Fires assigned to the county itself.
    fire_in = fires.groupBy("county_fips", "date_local").agg(
        F.count("*").alias("fire_detections"),
        F.sum(F.when(F.col("confidence_class") == "high", 1).otherwise(0)).alias("fire_detections_high_conf"),
        F.round(F.sum("frp_mw"), 1).alias("fire_frp_total_mw"),
        F.round(F.max("frp_mw"), 1).alias("fire_frp_max_mw"),
    )

    # Fires within NEAR_KM of the county centre (smoke can travel; this captures neighbours too).
    def hav(lat1, lon1, lat2, lon2):
        dlat = F.radians(lat2 - lat1)
        dlon = F.radians(lon2 - lon1)
        a = (
            F.pow(F.sin(dlat / 2), 2)
            + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlon / 2), 2)
        )
        return 2 * 6371.0 * F.asin(F.sqrt(a))

    fire_near = (
        fires.select("date_local", "latitude", "longitude", "frp_mw").alias("f")
        .crossJoin(F.broadcast(county.select("county_fips", "c_lat", "c_lon")).alias("c"))
        .where(hav(F.col("f.latitude"), F.col("f.longitude"), F.col("c.c_lat"), F.col("c.c_lon")) <= NEAR_KM)
        .groupBy("county_fips", "date_local")
        .agg(
            F.count("*").alias("fire_detections_within_100km"),
            F.round(F.sum("frp_mw"), 1).alias("fire_frp_within_100km_mw"),
        )
    )

    # PM2.5: average POCs/durations per site first, then across sites, so a site with
    # multiple monitors is not triple-weighted.
    pm_site = (
        spark.table(t("silver_pm25_daily"))
        .where(F.col("date_local").isin(days))
        .groupBy("county_fips", "site_id", "date_local")
        .agg(F.avg("pm25_ugm3").alias("site_pm25"), F.max("aqi").alias("site_aqi"))
    )
    pm = pm_site.groupBy("county_fips", "date_local").agg(
        F.round(F.avg("site_pm25"), 2).alias("pm25_mean_ugm3"),
        F.round(F.max("site_pm25"), 2).alias("pm25_max_site_ugm3"),
        F.max("site_aqi").alias("aqi_max"),
        F.countDistinct("site_id").alias("pm25_sites"),
    )

    # Weather: average stations in the county. Wind direction is averaged as a vector,
    # not arithmetically.
    wx = (
        spark.table(t("silver_weather_daily"))
        .where(F.col("date_local").isin(days))
        .groupBy("county_fips", "date_local")
        .agg(
            F.round(F.avg("tmax_c"), 1).alias("tmax_c"),
            F.round(F.avg("tmin_c"), 1).alias("tmin_c"),
            F.round(F.avg("prcp_mm"), 1).alias("prcp_mm"),
            F.round(F.avg("awnd_ms"), 1).alias("wind_mean_ms"),
            F.avg(F.sin(F.radians("wdf2_deg"))).alias("_s"),
            F.avg(F.cos(F.radians("wdf2_deg"))).alias("_c"),
            F.countDistinct("station").alias("weather_stations"),
        )
        .withColumn("wind_dir_deg", F.round((F.degrees(F.atan2("_s", "_c")) + 360) % 360, 0))
        .drop("_s", "_c")
    )

    keys = ["county_fips", "date_local"]
    fact = (
        grid.drop("c_lat", "c_lon")
        .join(fire_in, keys, "left")
        .join(fire_near, keys, "left")
        .join(pm, keys, "left")
        .join(wx, keys, "left")
        .fillna(
            0,
            subset=[
                "fire_detections",
                "fire_detections_high_conf",
                "fire_frp_total_mw",
                "fire_detections_within_100km",
                "fire_frp_within_100km_mw",
            ],
        )
        .withColumn("date_key", F.date_format("date_local", "yyyyMMdd").cast("int"))
        .withColumn("pm25_exceeds_daily_standard", F.col("pm25_mean_ugm3") > PM25_DAILY_STANDARD)
        .withColumn(
            "smoke_indicator",
            F.when(F.col("pm25_mean_ugm3").isNull(), "no_pm25_data")
            .when(
                (F.col("pm25_mean_ugm3") > PM25_DAILY_STANDARD)
                & (F.col("fire_detections_within_100km") > 0),
                "likely_smoke",
            )
            .when(F.col("pm25_mean_ugm3") > PM25_DAILY_STANDARD, "high_pm25_no_nearby_fire")
            .when(F.col("fire_detections_within_100km") > 0, "nearby_fire_pm25_ok")
            .otherwise("clear"),
        )
        .withColumn("has_pm25", F.col("pm25_sites").isNotNull())
        .withColumn("has_weather", F.col("weather_stations").isNotNull())
        .withColumn("sources_present", 1 + F.col("has_pm25").cast("int") + F.col("has_weather").cast("int"))
        .withColumn("_gold_built_at_utc", F.current_timestamp())
    )

    if not spark.catalog.tableExists(t("gold_fact_county_day")):
        fact.write.format("delta").saveAsTable(t("gold_fact_county_day"))
    else:
        # Update only when a measure actually changed, so reruns leave unchanged
        # county-days untouched apart from the audit timestamp.
        measures = [c for c in fact.columns if c not in keys + ["_gold_built_at_utc"]]
        changed = " OR ".join([f"NOT (g.{c} <=> n.{c})" for c in measures])
        (
            DeltaTable.forName(spark, t("gold_fact_county_day"))
            .alias("g")
            .merge(fact.alias("n"), "g.county_fips = n.county_fips AND g.date_local = n.date_local")
            .whenMatchedUpdateAll(condition=changed)
            .whenNotMatchedInsertAll()
            .execute()
        )

    # Recompute statewide KPIs over the complete Gold fact table, including dates
    # added after the original 2020 study window.
    (
        spark.table(t("gold_fact_county_day"))
        .groupBy("date_local")
        .agg(
            F.sum("fire_detections").alias("ca_fire_detections"),
            F.round(F.sum("fire_frp_total_mw"), 0).alias("ca_fire_frp_mw"),
            F.round(F.avg("pm25_mean_ugm3"), 2).alias("avg_county_pm25_ugm3"),
            F.sum(F.col("pm25_exceeds_daily_standard").cast("int")).alias("counties_over_standard"),
            F.sum(F.col("has_pm25").cast("int")).alias("counties_with_pm25"),
            F.sum((F.col("smoke_indicator") == "likely_smoke").cast("int")).alias("counties_likely_smoke"),
        )
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(t("gold_kpi_state_day"))
    )

    spark.sql(
        f"DELETE FROM {t('ops_gold_pending_dates')} "
        f"WHERE date_local IN ({','.join(repr(str(d)) for d in days)})"
    )
    spark.sql(f"OPTIMIZE {t('gold_fact_county_day')} ZORDER BY (date_local, county_fips)")

log_step(
    "gold_fact_county_day",
    "gold",
    spark.table(t("gold_fact_county_day")).count() if spark.catalog.tableExists(t("gold_fact_county_day")) else 0,
    started,
    {"dates_rebuilt": len(days), "dimension_end": str(dimension_end)},
)

/home/spark-6e4891ac-4bc0-4454-b47b-72/.ipykernel/62/command-6118707730931980-1423734058:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(step, layer, int(rows), float(elapsed), dt.datetime.utcnow(), json.dumps(extra or {}))]


[gold] gold_fact_county_day: 3,306 rows in 28.1s


In [0]:
display(
    spark.sql(f"""
        SELECT f.date_local, c.county_name, f.fire_detections_within_100km,
               f.pm25_mean_ugm3, f.wind_mean_ms, f.smoke_indicator
        FROM {t('gold_fact_county_day')} f
        JOIN {t('gold_dim_county')} c USING (county_fips)
        ORDER BY f.date_local DESC, f.pm25_mean_ugm3 DESC NULLS LAST
        LIMIT 20
    """)
)


date_local,county_name,fire_detections_within_100km,pm25_mean_ugm3,wind_mean_ms,smoke_indicator
2020-10-10,Mono,83,62.74,2.1,likely_smoke
2020-10-10,Inyo,16,51.81,2.4,likely_smoke
2020-10-10,San Diego,0,23.17,2.0,clear
2020-10-10,Siskiyou,5,21.84,2.4,nearby_fire_pm25_ok
2020-10-10,Kern,0,21.35,1.6,clear
2020-10-10,San Bernardino,1,18.12,5.4,nearby_fire_pm25_ok
2020-10-10,Riverside,0,16.96,2.9,clear
2020-10-10,Imperial,2,15.9,2.5,nearby_fire_pm25_ok
2020-10-10,Tehama,0,13.93,2.4,clear
2020-10-10,Ventura,1,12.49,3.2,nearby_fire_pm25_ok
